# Exploring the EBRAINS Knowledge Graph with fairgraph

[fairgraph](https://fairgraph.readthedocs.io) is a Python library that represents the metadata held in the
[EBRAINS Knowledge Graph](https://docs.kg.ebrains.eu) (KG) as Python objects, so that you can find, inspect,
and download EBRAINS research products from a script or a notebook.

This notebook is the hands-on part of the fairgraph session in the tutorial
*"The EBRAINS Knowledge Graph for data consumers"*. It is about **reading** metadata from the Knowledge Graph.
Creating and saving metadata is the subject of a separate tutorial for data *providers*, and is deliberately
not covered here.

Note that "data" is meant in a broad sense throughout: the KG describes datasets, but also computational
models, software, web services, live papers, and the publications and computations that connect them.

### How to use this notebook

Sections marked **&#11088;** are the core, and we will work through them together in the time available.
Sections marked **&#9711;** cover things you are less likely to need on your first day (custom queries,
local metadata snapshots, release status, error handling) and are there for you to explore afterwards.

1. &#11088; Setup and connection
2. &#11088; What is in the Knowledge Graph
3. &#11088; Finding things
4. &#11088; Following links
5. &#11088; Controlled terms
6. &#11088; One question, several kinds of research product
7. &#9711; Taking a local snapshot with `Collection`
8. &#9711; Custom queries
9. &#9711; Release status, spaces and ordering
10. &#9711; Loose ends, and where to go next

An appendix at the end has solutions to the exercises.

## &#11088; 1. Setup and connection

If you are running this notebook in the [EBRAINS Lab](https://lab.ebrains.eu) with a recent kernel from the
EBRAINS Software Distribution, fairgraph is already installed. Otherwise:

```
pip install fairgraph
```

### Authentication

Use of the KG API is restricted to EBRAINS users who have accepted the
[KG Terms of Use](https://docs.kg.ebrains.eu/search-terms-of-use.html). Please read them now if you have not
already done so. Every request carries an access token associated with your EBRAINS account, and `KGClient`
obtains one for you in whichever of these ways applies:

- **in the EBRAINS Lab**: automatically, from the notebook environment — there is nothing for you to do;
- **anywhere else**: the client prints the URL of a log-in page. Open it, log in to your EBRAINS account,
  close the tab, and carry on in the notebook;
- **from the environment**: set `KG_AUTH_TOKEN` before starting the notebook, and the client will find it;
- **explicitly**: `KGClient(token="eyJhbGci...")`, with a token copied from, for example, the
  [KG Editor](https://editor.kg.ebrains.eu). Such tokens are short-lived (around ten minutes), so you will
  need to repeat this from time to time as you work;
- **for an application**: `KGClient(client_id=..., client_secret=...)`, using a service account requested
  from EBRAINS Support.

### Which Knowledge Graph?

fairgraph talks to the *pre-production* KG by default, which is the right choice while you are developing
code that writes metadata. As data consumers we want the real thing, so we pass the production host
explicitly.

In [1]:
# Everything this notebook hard-codes, collected in one place so that it is easy to check or change.

KG_HOST = "core.kg.ebrains.eu"   # the production Knowledge Graph

# "Excitability profile of CA1 pyramidal neurons in APPPS1 Alzheimer disease mice
#  and control littermates"
DATASET_VERSION_ID = "bd5f91ff-e829-4b85-92eb-fc56991541f1"

# "Whole cell patch-clamp recordings of cerebellar Golgi cells"
EPHYS_DATASET_VERSION_ID = "17196b79-04db-4ea4-bb69-d20aab6f1d62"

EXAMPLE_DOI = "10.25493/YJFW-HPY"

THEME = "Alzheimer's disease"    # the scientific theme we follow in sections 5 and 6

In [2]:
import fairgraph
from fairgraph import KGClient
from fairgraph.utility import as_list
import fairgraph.openminds.core as omcore
import fairgraph.openminds.controlled_terms as terms

# Not every record in the public KG validates cleanly against the current schemas, and the resulting
# warnings would drown out everything else. We come back to error handling in section 10.
fairgraph.set_error_handling(None)

print("fairgraph version:", fairgraph.__version__)

fairgraph version: 0.13.3


In [3]:
client = KGClient(host=KG_HOST)

user = client.user_info()
print(f"Connected to {KG_HOST} as {user.given_name} {user.family_name}")

Connected to core.kg.ebrains.eu as Testing Davison


A few of the things shown further down need a newer fairgraph than the one you may have. The
EBRAINS-experimental kernel in the EBRAINS Software Distribution currently has **0.13.3**, and will have
**0.15.0** once the pending updates land.

Everything in the core sections works on 0.13.3. The handful of cells that need something newer check for it
and say so instead of failing, so you can run the whole notebook either way.

In [4]:
import inspect

try:
    from fairgraph import Regex      # fairgraph >= 0.15
    HAVE_REGEX = True
except ImportError:
    HAVE_REGEX = False

try:
    from fairgraph import Equals     # will be available once fairgraph 0.16.0 is released
    HAVE_EQUALS = True
except ImportError:
    HAVE_EQUALS = False

# by_name() gained these options, and learned to search every name-like property, in 0.15
HAVE_BY_NAME_OPTIONS = "case_sensitive" in inspect.signature(omcore.License.by_name).parameters

HAVE_SORT_PROPERTY = hasattr(omcore.Person, "sort_property")   # will be in fairgraph 0.16.0

for label, available in [
    ("Regex filters", HAVE_REGEX),
    ("Equals filters", HAVE_EQUALS),
    ("by_name(case_sensitive=...)", HAVE_BY_NAME_OPTIONS),
    ("sort_property", HAVE_SORT_PROPERTY),
]:
    print(f"{label:<30} {'yes' if available else 'no, those cells will be skipped'}")

Regex filters                  no, those cells will be skipped
Equals filters                 no, those cells will be skipped
by_name(case_sensitive=...)    no, those cells will be skipped
sort_property                  no, those cells will be skipped


## &#11088; 2. What is in the Knowledge Graph

The Knowledge Graph is a *semantic graph*: a collection of metadata **nodes** joined by typed **links**.
A node holds metadata about one aspect of a piece of research—a dataset, a version of that dataset, a
subject, a tissue sample, a recording session, a person, a licence—and a link records the relationship
between two of them, with a specific meaning: *this dataset version was produced by that recording activity*,
*that recording activity used this electrode*.

The types of node, and the properties each type may have, are defined by the
[openMINDS](https://openminds.docs.om-i.org) schemas.
fairgraph turns each openMINDS type into a Python class, and each property into an attribute.

The most visible nodes are the ones that represent *research products*. Let's see how many of each the KG
currently holds.

In [5]:
import fairgraph.openminds.publications as pubs

product_types = [
    omcore.DatasetVersion,
    omcore.ModelVersion,
    omcore.SoftwareVersion,
    omcore.MetaDataModelVersion,
    omcore.WebServiceVersion,
    pubs.LivePaperVersion,
    pubs.ScholarlyArticle,
]

for cls in product_types:
    print(f"{cls.__name__:<22} {cls.count(client):>6}")

DatasetVersion           1633


ModelVersion              358


SoftwareVersion           542


MetaDataModelVersion        5


WebServiceVersion          38


LivePaperVersion           29


ScholarlyArticle           35


Most of those class names end in "Version". openMINDS separates the *product* (`Dataset`, `Model`,
`Software`, ...) from each released *version* of it (`DatasetVersion`, `ModelVersion`,
`SoftwareVersion`, ...). The version carries what changes between releases—the files, the release date,
the version identifier—while the parent carries what does not, typically the name and description,
sometimes the authors.

**As a consumer you usually want the Version**, but you will frequently need to look at its parent too.
That turns out to matter a great deal when searching, as we will see in sections 3 and 4.

The classes are grouped into modules that mirror the openMINDS metadata domains:

| module | covers |
|---|---|
| `core` | research products, files, people, organizations, specimens |
| `controlled_terms` | the controlled vocabularies: species, cell types, diseases, techniques, ... |
| `sands` | brain atlases, anatomical locations |
| `ephys` | electrophysiology recordings in detail |
| `specimen_prep` | cell culture, surgery, tissue slicing |
| `stimulation` | stimulation protocols |
| `chemicals` | substances and mixtures |
| `computation` | simulations, data analyses, visualizations, workflows |
| `publications` | articles, books, live papers, learning resources |
| `neuroimaging` | *coming soon!* |

Each module can tell you what it contains:

In [6]:
import fairgraph.openminds.computation as computation

print(sorted(cls.__name__ for cls in computation.list_kg_classes()))

['DataAnalysis', 'DataCopy', 'Environment', 'GenericComputation', 'HardwareSystem', 'LaunchConfiguration', 'LocalFile', 'ModelValidation', 'Optimization', 'Simulation', 'SoftwareAgent', 'ValidationTest', 'ValidationTestVersion', 'Visualization', 'WorkflowExecution', 'WorkflowRecipe', 'WorkflowRecipeVersion']


## &#11088; 3. Finding things

There are four ways to get hold of a node: by its identifier, by its name, by filtering on its properties,
and by following a link from a node you already have. The first three are this section; links are section 4.

### By identifier

Every node has a UUID. If you have one, whether from a DOI landing page, from the Search UI, or from a colleague,
`from_id()` fetches the node directly.

In [7]:
dsv = omcore.DatasetVersion.from_id(DATASET_VERSION_ID, client)
dsv.show()

id                       https://kg.ebrains.eu/api/instances/bd5f91ff-e829-4b85-92eb-fc56991541f1
space                    dataset
type                     https://openminds.om-i.org/types/DatasetVersion
accessibility            KGProxy([ProductAccessibility], id="b2ff7a47-b349-48d7-8ce4-cf51868675f1")
data_types               KGProxy([SemanticDataType], id="f468ee45-37a6-4e71-8b70-0cbe66d367db")
description
digital_identifier       KGProxy([DOI, IdentifiersDotOrgID], id="df93b012-3bb6-4565-bce0-3bd2d3aed63b")
ethics_assessment        KGProxy([EthicsAssessment], id="a217a2f8-dcb8-4ca9-9923-517af2aebc5b")
experimental_approaches  KGProxy([ExperimentalApproach], id="411b233f-a054-4719-bbd5-e86f1302bc46")
full_documentation       KGProxy([DOI, File, ISBN, WebResource], id="a245c132-30ae-4ba8-a1d1-3a0f44d056aa")
full_name                Excitability profile of CA1 pyramidal neurons in APPPS1 Alzheimer disease mice and control li ...
keywords                 [KGProxy([ActionStatusType, AgeC

You can look at the same record in the KG Search UI by pasting the UUID after
`https://search.kg.ebrains.eu/instances/`:
[this one](https://search.kg.ebrains.eu/instances/bd5f91ff-e829-4b85-92eb-fc56991541f1).

Two things in that output are worth noting now:

1. many values are `KGProxy` objects. Those are *links* that have not been followed: see section 4.
2. `description` is empty, even though the Search UI shows a description for this dataset. That is the
   product/version split again, and section 4 shows the fix.

### By name

`by_name()` searches every name-like property the class has ()`name`, `full_name`, `short_name`,
`lookup_label`, `family_name`, `abbreviation`, `synonyms`) and by default wants an exact,
case-sensitive match.

In [8]:
dsv = omcore.DatasetVersion.by_name(
    "Excitability profile of CA1 pyramidal neurons in APPPS1 Alzheimer disease mice "
    "and control littermates",
    client,
)
print(dsv.uuid)

bd5f91ff-e829-4b85-92eb-fc56991541f1


The match can be relaxed. `match="contains"` (the property value contains the string you gave),
`match="within"` (the string you gave contains the property value) and `all=True` (return every match rather
than the first) work in any version. `case_sensitive=False` and `ignore_accents=True` (the latter also
treats special letters as their plain equivalents, so "ß" matches "ss") arrived in fairgraph 0.15, together
with the fix that makes `by_name()` search *every* name-like property a class has rather than stopping at
the first.

In [9]:
interneurons = terms.CellType.by_name("interneuron", client, match="contains", all=True)
print(f"{len(interneurons)} cell types whose name contains 'interneuron'")
print([ct.name for ct in interneurons[:5]])

if HAVE_BY_NAME_OPTIONS:
    # a licence's name lives in short_name, which older versions of by_name() did not search
    licence = omcore.License.by_name("cc-by-nc-sa-4.0", client, case_sensitive=False)
    print(f"\n{licence.short_name} - {licence.full_name}")
else:
    print("\ncase-insensitive by_name() needs fairgraph >= 0.15 - skipping")

8 cell types whose name contains 'interneuron'
['cerebellar interneuron', 'cholinergic interneuron', 'cortical interneuron', 'fast spiking interneuron', 'interneuron']

case-insensitive by_name() needs fairgraph >= 0.15 - skipping


### By filtering on properties

`list()` takes any property of the class as a keyword argument. A plain string is a **contains** match,
and matching is **always case-insensitive**.

In [10]:
for ds in omcore.DatasetVersion.list(client, full_name="patch-clamp"):
    print("-", ds.full_name)

- Whole cell patch-clamp recordings of cerebellar Golgi cells
- Whole cell patch-clamp recordings of cerebellar granule cells
- Whole cell patch-clamp recordings of cerebellar stellate cells


`property_names` tells you what you can filter on — useful at an interactive prompt, together with
`help(omcore.DatasetVersion)`.

In [11]:
print(omcore.DatasetVersion.property_names)

['accessibility', 'authors', 'behavioral_protocols', 'copyright', 'custodians', 'data_types', 'description', 'digital_identifier', 'ethics_assessment', 'experimental_approaches', 'full_documentation', 'full_name', 'funding', 'homepage', 'how_to_cite', 'input_data', 'is_alternative_version_of', 'is_new_version_of', 'keywords', 'license', 'other_contributions', 'preparation_designs', 'protocols', 'related_publications', 'release_date', 'repository', 'short_name', 'studied_specimens', 'study_targets', 'support_channels', 'techniques', 'version_identifier', 'version_innovation']


For more control over the match, pass a `Regex` instead of a plain string (fairgraph 0.15 and later).
The pattern is applied as a
*search*, so an unanchored pattern matches anywhere in the value; `^` and `$` anchor it. fairgraph checks
that the pattern compiles before sending it, so a typo raises `ValueError` rather than quietly matching
nothing. Note that the KG's matching is case-insensitive whatever your pattern says — to match case exactly,
filter the results in Python.

In [12]:
if HAVE_REGEX:
    datasets = omcore.DatasetVersion.list(client, full_name=Regex("^Whole cell patch-clamp"))
    for ds in datasets:
        print("-", ds.full_name)
else:
    print("Regex filters need fairgraph >= 0.15 - skipping")

Regex filters need fairgraph >= 0.15 - skipping


`Equals` is for when the *whole* value must match: `short_name=Equals("FOO")` finds a dataset called "FOO"
but not one called "FOO-BAR". It is in development and not in a release yet, so the cell below will probably
skip itself — the difference it makes is worth knowing about all the same.

In [13]:
print("version identifier contains 'v1':  ",
      omcore.DatasetVersion.count(client, version_identifier="v1"))

if HAVE_EQUALS:
    print("version identifier is exactly 'v1':",
          omcore.DatasetVersion.count(client, version_identifier=Equals("v1")))
else:
    print("Equals is not in a released version yet - skipping")

version identifier contains 'v1':   955
Equals is not in a released version yet - skipping


### Filtering across a link

Often the value you want to search for is not on the node itself but on a node it links to. The DOI of a
dataset version, for instance, is not a string property: `digital_identifier` is a *link* to a `DOI` node,
and the DOI string lives in that node's `identifier` property.

Join property names with a double underscore to filter across the link:

In [14]:
found = omcore.DatasetVersion.list(client, digital_identifier__identifier=EXAMPLE_DOI)
for item in found:
    print(item.uuid, "-", item.full_name)

bd5f91ff-e829-4b85-92eb-fc56991541f1 - Excitability profile of CA1 pyramidal neurons in APPPS1 Alzheimer disease mice and control littermates


This works to any depth, and is how you would find, say, every dataset whose authors are affiliated with a
given institute.

In [15]:
cnrs = omcore.Organization.by_name("French National Centre for Scientific Research", client)
datasets = omcore.DatasetVersion.list(client, authors__affiliations__member_of=cnrs)

print(f"{cnrs.full_name} ({cnrs.uuid})")
print(f"{len(datasets)} dataset versions with an author affiliated there")
for item in datasets:
    print("  ", item.uuid)

French National Centre for Scientific Research (31259b06-91d0-4ad8-acfd-303fc9ed613b)
2 dataset versions with an author affiliated there
   a1c940cc-4777-417e-9326-dd6584d6c71f
   8ddf749f-fb1d-4d16-acc3-fbde91b90e24


Treat the result as a lower bound. Only a minority of `Person` records have their affiliation
recorded.

### A trap worth knowing about

Remember that the name and description often live on the parent `Dataset` rather than on the
`DatasetVersion`. Searching the version alone therefore misses records:

In [16]:
on_the_version = omcore.DatasetVersion.list(client, full_name="Alzheimer")
on_the_parent = omcore.DatasetVersion.list(client, is_version_of__full_name="Alzheimer")

print(f"matching the version's own name: {len(on_the_version)}")
print(f"matching the parent's name:      {len(on_the_parent)}")

combined = {dsv.uuid: dsv for dsv in on_the_version + on_the_parent}
print(f"combined, after removing duplicates: {len(combined)}")

matching the version's own name: 1
matching the parent's name:      5
combined, after removing duplicates: 5


This is one reason the Search UI returns more hits for a word than a naive fairgraph query does: it searches
several fields at once, including descriptions. When you need the same breadth, run the queries you need and
combine the results in Python, keyed on `uuid` as above.

Two more things that bite newcomers:

- **filtering is not hierarchy-aware.** Filtering on the brain region "hippocampus" will *not* return records
  annotated with "hippocampus CA1". You have to ask for the terms you want, or walk the hierarchy yourself.
- **whitespace is significant** even though case is not.

### Counting and paging

`list()` returns the first 100 matches by default. Use `size` and `from_index` to page through more, and
`count()` to ask how many there are in total without fetching them.

In [17]:
print("released dataset versions in total:", omcore.DatasetVersion.count(client))

print("with 'recordings' in the name:   ",
      omcore.DatasetVersion.count(client, full_name="recordings"))

page1 = omcore.DatasetVersion.list(client, full_name="recordings", size=3, from_index=0)
page2 = omcore.DatasetVersion.list(client, full_name="recordings", size=3, from_index=3)
for page_number, page in enumerate([page1, page2], start=1):
    print(f"\npage {page_number}:")
    for ds in page:
        print("  ", ds.full_name)

released dataset versions in total: 1633


with 'recordings' in the name:    13



page 1:
   1024-channel electrophysiological recordings in macaque V1 and V4 during resting state
   Database of paired recordings in hippocampal slices
   Electrophysiological recordings of striatal cholinergic interneuron

page 2:
   Electrophysiological recordings of striatal fast-spiking interneurons
   Electrophysiological recordings of striatal low threshold-spiking interneuron
   Electrophysiological recordings of striatal projection neurons


> **Exercise 1.** Find every dataset version with "patch-clamp" in its name, and print the version
> identifier and release date of each one. (Solutions are in the appendix.)

## &#11088; 4. Following links

**Links are not followed automatically.** If they were, fetching one dataset would drag in half the graph.
Instead, an unfollowed link appears as a placeholder:

- a `KGProxy` stands for a link to one known node;
- a `KGQuery` stands for a link that has to be resolved by running a query — this is what you get for
  *reverse* links.

Both have a `resolve()` method.

In [18]:
dsv = omcore.DatasetVersion.from_id(DATASET_VERSION_ID, client)

print(dsv.license)                     # a placeholder
licence = dsv.license.resolve(client)  # one request to the KG
print(licence.short_name, "-", licence.full_name)

KGProxy([License, WebResource], id="64c1704b-db12-4b29-9541-b9143d081044")


CC-BY-4.0

 - Creative Commons Attribution 4.0 International


### Forward and reverse properties

The *forward* properties of a class are the ones openMINDS defines: `DatasetVersion.license`,
`DatasetVersion.authors`. fairgraph adds *reverse* properties, which expose the links that point **at** this
node — they are not in the schema, but they are often exactly what you want.

In [19]:
print("forward:", [p.name for p in omcore.DatasetVersion.properties][:10], "...")
print("reverse:", [p.name for p in omcore.DatasetVersion.reverse_properties])

forward: ['accessibility', 'authors', 'behavioral_protocols', 'copyright', 'custodians', 'data_types', 'description', 'digital_identifier', 'ethics_assessment', 'experimental_approaches'] ...
reverse: ['comments', 'has_parts', 'is_input_to', 'is_old_version_of', 'is_part_of', 'is_version_of', 'learning_resources', 'publication']


`is_version_of` is one of those reverse properties: `Dataset.has_versions` points forwards, and fairgraph
gives you the way back. Reverse properties are hidden by `show()` unless you ask for them, with
`dsv.show(include_empty_properties=True)`; `list()` accepts `with_reverse_properties=True`.

In [20]:
parent = dsv.is_version_of.resolve(client)
print(type(parent).__name__, "-", parent.full_name)

Dataset - Excitability profile of CA1 pyramidal neurons in APPPS1 Alzheimer disease mice and control littermates


### Following several links at once

Resolving links one at a time means one network request each, which quickly dominates the running time. If
you know in advance which links you want, pass a `follow_links` dict — nested to mirror the path through the
graph — and the KG returns the whole tree in a single request. It works both on `resolve()` and directly on
`from_id()` and `list()`.

In [21]:
print("# before")
print("license:      ", type(dsv.license).__name__)
print("is_version_of:", type(dsv.is_version_of).__name__)

dsv = omcore.DatasetVersion.from_id(
    DATASET_VERSION_ID,
    client,
    follow_links={"license": {}, "is_version_of": {"authors": {}}},
)

print("\n# after")
print("license:      ", type(dsv.license).__name__)
print("is_version_of:", type(dsv.is_version_of).__name__)

# before
license:       KGProxy
is_version_of: KGQuery



# after
license:       License
is_version_of: list


Note that this request followed a reverse link (`is_version_of`) and then a forward one (`authors`); any
combination works. The authors are now available without further requests:

In [22]:
for parent in as_list(dsv.is_version_of):
    for person in as_list(parent.authors):
        print("-", person.full_name)

- Ana Rita Salgueiro-Pereira
- Hélène Marie


`as_list()` is worth knowing about: a property that *may* hold several values holds a bare object when there
is only one, and `None` when there is none, so wrapping it keeps your loops simple.

Two caveats on `follow_links`: the KG limits the length of a query, so a very large or very deep tree can
fail with a server error; and following a link you do not need is wasted traffic.

### Inherited properties

Back to that empty description. `DatasetVersion` and the other version classes have `get_description()`,
`get_full_name()`, `get_short_name()` and `get_authors()` (or `get_developers()`, for software and models),
which fall back to the parent record when the version itself has no value.

In [23]:
ephys_dsv = omcore.DatasetVersion.from_id(EPHYS_DATASET_VERSION_ID, client)

print("on the version:", repr(ephys_dsv.description))
print()
print("inherited:     ", ephys_dsv.get_description(client)[:300], "...")

on the version: ''



inherited:      The Golgi cells, together with granule cells and mossy fibers, form a neuronal microcircuit regulating information transfer at the cerebellum input stage. In order to further investigate the Golgi cells properties and their excitatory synapses, whole-cell patch-clamp recordings were performed on acu ...


### Where a query starts determines what it costs

Both of the following find the files belonging to a dataset version, and both cross the same two links:

```python
# starts at one DatasetVersion, and traverses outwards to its files
dsv = omcore.DatasetVersion.from_id(dsv_id, client, follow_links={"repository": {"files": {}}})
files = dsv.repository.files

# starts at every File in the Knowledge Graph, and filters back to that dataset version
files = omcore.File.list(client, file_repository__contains_content_of=dsv)
```

They return the same files. For the dataset version used below, the first takes about 1.5 seconds and the
second closer to three minutes.

The difference is not that one follows a link while the other filters on one — both do both. It is where
each query is *rooted*. `from_id` starts from a single known node, so there is one node to expand outwards
from. `File.list` starts from the whole `File` class, close to a million nodes at the time of writing, and
narrows down. The cost follows the size of the starting set rather than the length of the path, and it has
nothing to do with the size of your dataset: this dataset version has 19 files and still pays the full
price.

So whenever you already hold a node, start the query from it.

In [24]:
ephys_dsv = omcore.DatasetVersion.from_id(
    EPHYS_DATASET_VERSION_ID, client, follow_links={"repository": {"files": {}}}
)
files = as_list(ephys_dsv.repository.files)

print(f"{len(files)} files in this dataset version")
for f in files[:5]:
    print(f"  {f.name}")

19 files in this dataset version
  18419-A_0001_IV.abf
  130519_0020_EPSC.abf
  18419-D_0015_IV.abf
  HBP-DataDescriptor_PatchGolgCs.txt
  30419_0015_EPSC.abf


> **Exercise 2.** Which five files in that dataset version are the largest? (`File.storage_size` holds a
> `QuantitativeValue`, whose `value` attribute is a number.)

## &#11088; 5. Controlled terms

Many of the most useful properties — `study_targets`, `techniques`, `experimental_approaches`,
`data_types` — do not hold free text. They link to nodes from a controlled vocabulary, so that everyone
annotating a dataset with a species or a disease uses the same term, with the same definition.

Those vocabularies live in `fairgraph.openminds.controlled_terms`, and there are two ways to look a term up.
openMINDS ships the vocabularies with the library, so if you call `by_name()` **without** a client, no
network request is made at all and you get back an object with a semantic IRI. Pass a client and you get the
corresponding node in the KG, identified by a UUID.

In [25]:
from fairgraph.openminds.controlled_terms import Species

print(Species.by_name("Mus musculus").id)            # from the built-in instance library, offline
print(Species.by_name("Mus musculus", client).id)    # the node in the Knowledge Graph

https://openminds.om-i.org/instances/species/musMusculus
https://kg.ebrains.eu/api/instances/d9875ebd-260e-4337-a637-b62fed4aa91d


The commonest terms are also available as class attributes, which is convenient and typo-proof:

In [26]:
rat = Species.rattus_norvegicus
print(rat.name, "|", rat.id)
print(rat.definition)

Rattus norvegicus | https://openminds.om-i.org/instances/species/rattusNorvegicus
The species *Rattus norvegicus* (brown rat) belongs to the family of *muridae* (murids).


Which vocabularies does a property accept? Ask the property itself. This saves a lot of guessing.

In [27]:
prop = omcore.DatasetVersion.get_property("study_targets")
print(sorted(cls.__name__ for cls in prop.types))

['AuditoryStimulusType', 'BiologicalOrder', 'BiologicalSex', 'BreedingType', 'CellCultureType', 'CellType', 'CustomAnatomicalEntity', 'Disease', 'DiseaseModel', 'ElectricalStimulusType', 'GeneticStrainType', 'GustatoryStimulusType', 'Handedness', 'MolecularEntity', 'OlfactoryStimulusType', 'OpticalStimulusType', 'Organ', 'OrganismSubstance', 'OrganismSystem', 'ParcellationEntity', 'ParcellationEntityVersion', 'Species', 'SubcellularEntity', 'TactileStimulusType', 'TermSuggestion', 'TissueSampleType', 'UBERONParcellation', 'VisualStimulusType']


"Alzheimer's disease" should be a `Disease`, so let's find that term in the KG and use it as a filter.

In [28]:
disease = terms.Disease.by_name(THEME, client)
disease.show()

id                             https://kg.ebrains.eu/api/instances/161baa02-4e08-4cf2-a641-81cf323cc15d
space                          controlled
type                           https://openminds.om-i.org/types/Disease
name                           Alzheimer's disease
preferred_ontology_identifier  http://purl.obolibrary.org/obo/DOID_10652
synonyms                       Alzheimer disease


In [29]:
studied_by = omcore.DatasetVersion.list(client, study_targets=disease)
print(f"{len(studied_by)} dataset versions have '{disease.name}' as a study target")

11 dataset versions have 'Alzheimer's disease' as a study target


And now the other direction. Because `study_targets` points at the term, fairgraph gives the term a reverse
property pointing back: `studied_in`. Nothing has been fetched for it, though — it holds a `KGQuery`, a query
waiting to be run. That is unavoidable here, because many different types of node can point at a term, and
the KG has to be asked about each of them separately. The `KGQuery` knows which:

In [30]:
from collections import Counter

print(type(disease.studied_in).__name__)
print(sorted(cls.__name__ for cls in disease.studied_in.classes))

KGQuery
['CellPatching', 'CranialWindowPreparation', 'DataAnalysis', 'DataCopy', 'DatasetVersion', 'ElectrodePlacement', 'GenericComputation', 'Model', 'ModelValidation', 'Optimization', 'ProtocolExecution', 'RecordingActivity', 'Simulation', 'StimulationActivity', 'TissueCulturePreparation', 'TissueSampleSlicing', 'ValidationTest', 'Visualization']


Note what is in that list: not only dataset versions, but models, simulations, data analyses, and the
experimental activities — cell patching, recording, tissue slicing — that produced the data. A controlled
term is a hub connecting everything in EBRAINS that shares a scientific subject, which is a far better way
to find related work than guessing keywords in titles.

`disease.studied_in.resolve(client)` would fetch all of it, and `disease.studied_in.count(client)` would
just count it — but either one queries every type in that list in turn, so for a well-used term it is slow.
When you know which types you care about, ask for those directly, which is exactly what the next section
does.

## &#11088; 6. One question, several kinds of research product

Let's put the pieces together: take one scientific question — *what has EBRAINS got on this disease?* — and
answer it across several types of node, then build a small table of the results.

`study_targets` is a property of `DatasetVersion`, of `Model`, and of the computation types, so the same
filter works on all of them. Expect some of the counts to be zero: that is information too, and it tells
you something about what the community has shared so far.

In [31]:
classes_of_interest = [
    omcore.DatasetVersion,
    omcore.Model,
    computation.Simulation,
    computation.DataAnalysis,
    computation.ValidationTest,
]

products = []
for cls in classes_of_interest:
    found = cls.list(client, study_targets=disease)
    print(f"{cls.__name__:<16} {len(found)}")
    products.extend(found)

DatasetVersion   11


Model            1


Simulation       0


DataAnalysis     0


ValidationTest   0


In [32]:
def label(obj):
    """The best available human-readable name.

    Prefers a full name to a short one. For version types, get_full_name() and get_short_name()
    fall back to the parent record when the version itself has no name. Last resort is the UUID.
    """
    for attribute in ("get_full_name", "get_short_name"):
        if hasattr(obj, attribute):
            try:
                value = getattr(obj, attribute)(client)
            except Exception:       # e.g. no parent record to inherit from
                value = None
            if value:
                return value
    for attribute in ("full_name", "name", "lookup_label", "short_name"):
        value = getattr(obj, attribute, None)
        if value:
            return value
    return obj.uuid


def doi_of(obj):
    identifier = getattr(obj, "digital_identifier", None)
    if identifier is None:
        return ""
    identifier = as_list(identifier)[0]
    if hasattr(identifier, "resolve"):
        identifier = identifier.resolve(client)
    return getattr(identifier, "identifier", "") or ""


rows = [
    {
        "type": type(p).__name__,
        "name": label(p),
        "released": str(getattr(p, "release_date", "") or ""),
        "doi": doi_of(p),
    }
    for p in products
]

for row in sorted(rows, key=lambda r: (r["type"], r["name"])):
    print(f"{row['type']:<16} {row['released']:<12} {row['name'][:60]:<62} {row['doi']}")

DatasetVersion                AI-Mind Project for Dementia Prediction with time series, ge   https://doi.org/10.25493/YHJF-4MT
DatasetVersion   2018-05-22   Brainwide distribution and variance of amyloid-beta deposits   https://doi.org/10.25493/G6CQ-D4D
DatasetVersion                Characteristics of electroencephalographical, MRI, and neuro   https://doi.org/10.25493/8MVP-NPN
DatasetVersion                Characteristics of electroencephalographical, MRI, and neuro   https://doi.org/10.25493/FWHQ-6W5
DatasetVersion                European Prevention of Alzheimer’s Dementia Longitudinal Coh   https://doi.org/10.25493/9SVQ-3X
DatasetVersion   2020-04-21   Excitability profile of CA1 pyramidal neurons in APPPS1 Alzh   https://doi.org/10.25493/YJFW-HPY
DatasetVersion   2023-04-11   Investigating cellular diversity in a novel Alzheimer’s dise   https://doi.org/10.25493/SZ0M-EE6
DatasetVersion   2022-12-31   MiniVess: A dataset of rodent cerebrovasculature from in viv   https://doi.org/10.

That is a plain list of dicts, which is all you need to print a table — and also exactly what
`pandas.DataFrame(rows)` wants, if you would rather have a DataFrame to sort and filter:

```python
import pandas as pd
pd.DataFrame(rows)
```

Not every product type is annotated with study targets, though. Software, in particular, is usually found by
name — the same `list()` call, a different property:

In [33]:
for sv in omcore.SoftwareVersion.list(client, is_version_of__full_name="neuron", size=5):
    print(f"- {label(sv)} ({sv.version_identifier})")

- CoreNeuron (0.16)
- Hodgkin Huxley Neuron Builder (1.3.0)
- Hodgkin-Huxley Neuron Builder (2.3.5)
- Hodgkin-Huxley Neuron Builder (2.2)
- NEURON (8.2.6)


### Getting the data itself

The KG stores metadata; the data files live in object storage, and the KG records where. We already have the
files of the ephys dataset version from section 4, so let's look at them properly and fetch the smallest one.

In [34]:
def size_in_bytes(f):
    if f.storage_size and f.storage_size.value:
        return f.storage_size.value
    return float("inf")

for f in sorted(files, key=size_in_bytes)[:5]:
    print(f"{size_in_bytes(f):>12,.0f} bytes  {f.name}")

       5,337 bytes  HBP-DataDescriptor_PatchGolgCs.txt
     143,466 bytes  Licence (CCBY-NC-SA).pdf
     647,168 bytes  18419-A_0001_IV.abf
     647,168 bytes  18419-D_0015_IV.abf
     647,168 bytes  130519_0014_IV.abf


`download()` takes the local path first and the client second. Passing `accept_terms_of_use=True` records
that you have read the KG Terms of Use — fairgraph otherwise stops and asks.

In [35]:
smallest = min(files, key=size_in_bytes)

local_path = smallest.download(".", client, accept_terms_of_use=True)
print("downloaded to", local_path)

downloaded to HBP-DataDescriptor_PatchGolgCs.txt


A whole dataset version can be downloaded as a zip archive in one call. **Don't run this during the
tutorial** — some EBRAINS datasets are very large indeed:

```python
local_file, repository_url = ephys_dsv.download("./downloads", client, accept_terms_of_use=True)
```

### Citing what you found

Finally, the `publications` module. `ScholarlyArticle` knows how to assemble a citation string, walking the
links to the issue, volume and journal for you.

In [36]:
articles = pubs.ScholarlyArticle.list(client, name="hippocamp", size=3)

for article in articles:
    try:
        print("-", article.get_citation_string(client))
    except Exception as err:
        print(f"- {article.name} (could not build a citation: {err})")

- Addolorata Marasco, Emiliano Spera, Vittorio De Falco, Annalisa Iuorio, Carmen A. Lupascu, Sergio M.G. Solinas & Michele Migliore (2023). An Adaptive Generalized Leaky Integrate-and-Fire model for hippocampal CA1 pyramidal neurons and interneurons. Bulletin of Mathematical Biology, 85: 109.


- John P. McCauley, Maurice A. Petroccione, Lianna Y. D'brant, Gabrielle C. Todd, Nurat Affinnih, Justin J. Wisnoski, Shergil Zahid, Swasti Shree, Alioscka A. Sousa, Rose M. De Guzman, Rosanna Migliore, Alexey Brazhe, Richard D. Leapman, Alexander Khmaladze, Alexey Semyanov, Damian G. Zuloaga, Michele Migliore & Annalisa Scimemi (2020). Circadian Modulation of Neurons and Astrocytes Controls Synaptic Plasticity in Hippocampal Area CA1. Cell Reports, 33: 108255.


- Carmen A. Lupascu, Annunziato Morabito, Federica Ruggeri, Chiara Parisi, Domenico Pimpinella, Rocco Pizzarelli, Giovanni Meli, Silvia Marinelli, Enrico Cherubini, Antonino Cattaneo & Michele Migliore (2020). Computational Modeling of Inhibitory Transsynaptic Signaling in Hippocampal and Cortical Neurons Expressing Intrabodies Against Gephyrin. Frontiers in Cellular Neuroscience, 14: 173.


> **Exercise 3.** Pick a brain region from `fairgraph.openminds.sands.ParcellationEntity` and report how
> many research products of each type study it.

That is the end of the guided part. Everything below is for later.

## &#9711; 7. Taking a local snapshot with `Collection`

An analysis that depends on metadata is only reproducible if you can say what the metadata *were* when you
ran it. A `Collection` gathers nodes and writes them to a local JSON-LD file, which you can keep beside your
code and reload later with no network access and no token.

In [37]:
from fairgraph import Collection

plain = omcore.DatasetVersion.from_id(EPHYS_DATASET_VERSION_ID, client)

snapshot = Collection(plain)
print(snapshot.statistics())

Counter({'DatasetVersion': 1})


One node, even though that dataset version links to a licence, a DOI, a repository, authors and much else.
`Collection` adds the nodes it is given *and* the nodes they link to — but only where the link has actually
been resolved. An unresolved `KGProxy` is written out as a bare reference to an id, not as content.

So the snapshot you get is exactly as complete as the object you put in it. Fetch with `follow_links`, and
the linked metadata come too:

In [38]:
detailed = omcore.DatasetVersion.from_id(
    EPHYS_DATASET_VERSION_ID,
    client,
    follow_links={"license": {}, "digital_identifier": {}, "repository": {}},
)

snapshot = Collection(detailed)
print(snapshot.statistics())

Counter({'DatasetVersion': 1, 'DOI': 1, 'License': 1, 'FileRepository': 1})


In [39]:
snapshot.save("ephys_dataset_snapshot.jsonld")

reloaded = Collection()
try:
    reloaded.load("ephys_dataset_snapshot.jsonld")   # no client, no network
    print(reloaded.statistics())
except KeyError as err:
    # fairgraph < 0.15 cannot reload a collection that refers to nodes it does not itself hold,
    # which is almost any snapshot, since a fetched object keeps unresolved links
    print(f"reloading a snapshot needs fairgraph >= 0.15 (it looked for {err})")

reloading a snapshot needs fairgraph >= 0.15 (it looked for 'https://kg.ebrains.eu/api/instances/f468ee45-37a6-4e71-8b70-0cbe66d367db')


Reloading needs fairgraph 0.15 or later: earlier versions raise `KeyError` on any snapshot that refers to a
node it does not itself contain, which is almost all of them.

One thing to be aware of: a `Collection` walks *forward* links only. Resolving a reverse property such as
`is_version_of` does not pull the parent record into the snapshot, so if you need it, add it to the
collection yourself.

`save()` also takes `individual_files=True`, to write one file per node into a directory, and
`include_empty_properties=True` if you want the unset properties written out as nulls.

A single node can be dumped and reloaded the same way, with `dump()` and `load()`:

```python
detailed.dump("one_dataset_version.jsonld")
again = omcore.DatasetVersion.load("one_dataset_version.jsonld")
```

## &#9711; 8. Custom queries

Everything we have done so far has gone through `list()` and `from_id()`, which build KG queries for you.
Those queries are JSON-LD documents, and they are the same kind of thing the
[Query Builder](https://query.kg.ebrains.eu) produces. Sometimes it is useful to see one, or to run one you
wrote yourself.

In [40]:
import json

query = omcore.DatasetVersion.generate_query(
    client,
    space=None,
    filters={"study_targets": disease},
    follow_links={"license": {}},
)

print(json.dumps(query, indent=2)[:1200], "...")

{
  "@context": {
    "@vocab": "https://core.kg.ebrains.eu/vocab/query/",
    "query": "https://schema.hbp.eu/myQuery/",
    "propertyName": {
      "@id": "propertyName",
      "@type": "@id"
    },
    "merge": {
      "@type": "@id",
      "@id": "merge"
    },
    "path": {
      "@id": "path",
      "@type": "@id"
    }
  },
  "meta": {
    "type": "https://openminds.om-i.org/types/DatasetVersion",
    "description": "Automatically generated by fairgraph"
  },
  "structure": [
    {
      "path": "@id",
      "filter": {
        "op": "EQUALS",
        "parameter": "id"
      }
    },
    {
      "path": "https://core.kg.ebrains.eu/vocab/meta/space",
      "propertyName": "query:space"
    },
    {
      "path": "@type"
    },
    {
      "path": "https://openminds.om-i.org/props/accessibility",
      "propertyName": "accessibility",
      "structure": [
        {
          "path": "@id"
        },
        {
          "path": "@type"
        }
      ],
      "singleValue": "FIRST

You can hand such a definition straight to the client, which returns the raw JSON-LD rather than fairgraph
objects:

In [41]:
page = client.query(query=query, size=3, release_status="released")

print("total matches:", page.total)
for item in page.data:
    print("-", item["@id"])

total matches: 11
- https://kg.ebrains.eu/api/instances/685b5844-7627-459c-a184-be1bab132c9d
- https://kg.ebrains.eu/api/instances/e3bcc696-9f24-46a9-ae21-4761caafb068
- https://kg.ebrains.eu/api/instances/c404e7f6-9a18-4a5c-ae3c-27f159f4f96a


A query you want to keep — or share with colleagues, or use from the Query Builder — can be stored in the KG
under a label and run by reference:

```python
client.store_query("my-alzheimer-datasets", query, space="myspace")
definition = client.retrieve_query("my-alzheimer-datasets")
page = client.query(query=definition, use_stored_query=True)
```

To write a query definition from scratch, `fairgraph.queries` has the building blocks: `Query`,
`QueryProperty`, `Filter`, and `PathElement` for the awkward cases, where one step of a path has to be
traversed in reverse (`reverse=True`) or restricted to one node type (`type_filter=...`).

## &#9711; 9. Release status, spaces and ordering

**Release status.** KG metadata are either "in progress" or "released". Everything in this notebook has used
the default, `release_status="released"`, which is almost always what a consumer wants: it is the curated,
publicly citable view. `"in progress"` shows work in preparation that you have permission to see, and
`"any"` shows both.

**Spaces.** The KG is divided into spaces with their own access permissions, and a query covers every space
you can read unless you narrow it with `space=`.

In [ ]:
print(client.spaces(names_only=True))

If a query consistently returns an empty list where you expected results, insufficient permissions are the
likeliest cause; see the [permissions documentation](https://fairgraph.readthedocs.io/en/latest/permissions.html).

**Ordering.** The KG has two APIs, and `list()` picks one for you: with filters or `follow_links` it uses the
query API, and otherwise the core API, which is faster but does **not** sort. Pass `api="query"` when you want
an unfiltered list in a predictable order.

In [43]:
licences = omcore.License.list(client, api="query", size=5)
print("sorted:  ", [lic.full_name[:24] for lic in licences])

# not every class is sorted, though - Person is one that is not
people = omcore.Person.list(client, api="query", size=5)
print("unsorted:", [p.family_name for p in people])

if HAVE_SORT_PROPERTY:
    print("License is sorted by:", omcore.License.sort_property)

sorted:   ['Apache License 2.0', "BSD 2-Clause 'Simplified", "BSD 3-Clause 'New' or 'R", "BSD 4-Clause 'Original' ", 'CeCILL Free Software Lic']


unsorted: ['Smiriglia', 'Davison', 'Appukuttan', 'Schiffer', 'Svoboda']


Sorting is ascending and case-insensitive, on exactly one property (the KG allows no more) and nodes with
no value for it come first. Which property that is depends on the class, and some classes are not sorted at
all: `Person` is the one you are most likely to notice, as the cell above shows. Naming the property
explicitly, as `sort_property`, and making `Person` sort by `family_name`, is in development and not in a
release yet, so the last line above will probably be skipped.

**Caching.** The client caches nodes it has fetched, so resolving the same link twice costs one request.
Pass `use_cache=False` to `resolve()` if you need to be sure you are seeing the current state.

## &#9711; 10. Loose ends, and where to go next

### Error handling

We turned validation messages off at the top of this notebook. In your own code you can choose how strictly
schema violations are reported, globally, per module, or per class: `"error"` raises an Exception, `"warning"` warns,
`"log"` sends messages to the logging system, and `None` is silent.

```python
fairgraph.set_error_handling("warning")              # everything
fairgraph.openminds.core.set_error_handling("log")   # one module
omcore.DatasetVersion.set_error_handling(None)       # one class
```

Reading public metadata is the case where silence is usually right: you did not create these records and
cannot fix them. When you start *writing* metadata, turn it back on, and `obj.validate()` will tell you
what is wrong with a node before you try to save it.

### openMINDS v4 and v5

This notebook has used openMINDS v4 classes, which are what `fairgraph.openminds.core` gives you and what
the production and pre-production Knowledge Graphs serve. v5 classes also exist, as
`fairgraph.openminds.v5.core` and so on, and a client can be told to use them with
`KGClient(..., openminds_version="v5")` in fairgraph 0.15 and later — but v5 support is **experimental**,
the KG migration is still in progress, and consumers should stay on v4 for now. See the
[metadata domains documentation](https://fairgraph.readthedocs.io/en/latest/modules.html).

### Where to go next

- the [fairgraph documentation](https://fairgraph.readthedocs.io), in particular the
  [querying guide](https://fairgraph.readthedocs.io/en/latest/queries.html)
- `help(omcore.DatasetVersion)` and `DatasetVersion.property_names`, at any Python prompt
- the [advanced tutorial notebook](https://github.com/HumanBrainProject/fairgraph/blob/master/examples/notebooks/advanced_tutorial.ipynb)
  in this repository, which goes further into the KG's structure
- the equivalent [tutorial for JavaScript](https://github.com/apdavison/ebrains-kg-tutorial-javascript)
- questions to support@ebrains.eu; bugs and feature requests to the
  [issue tracker](https://github.com/HumanBrainProject/fairgraph/issues)
- and, for creating and saving metadata of your own, the companion tutorial for **data providers**

## Appendix: exercise solutions

In [44]:
# Exercise 1: the "patch-clamp" dataset versions

found = omcore.DatasetVersion.list(client, full_name="patch-clamp")
for item in found:
    print(f"{item.version_identifier:<6} {item.release_date}  {item.full_name}")

v1     2020-03-26  Whole cell patch-clamp recordings of cerebellar Golgi cells
v1     2020-03-24  Whole cell patch-clamp recordings of cerebellar granule cells
v1     2020-03-25  Whole cell patch-clamp recordings of cerebellar stellate cells


In [45]:
# Exercise 2: the five largest files in a dataset version

dsv = omcore.DatasetVersion.from_id(
    EPHYS_DATASET_VERSION_ID, client, follow_links={"repository": {"files": {}}}
)

def size_of(f):
    return f.storage_size.value if (f.storage_size and f.storage_size.value) else 0

for f in sorted(as_list(dsv.repository.files), key=size_of, reverse=True)[:5]:
    print(f"{size_of(f):>14,.0f} bytes  {f.name}")

     1,555,456 bytes  130519_0020_EPSC.abf
     1,555,456 bytes  30419_0015_EPSC.abf
     1,555,456 bytes  130519_0022_EPSC.abf
     1,555,456 bytes  250319_0008_EPSC.abf
     1,555,456 bytes  100419_0027_EPSC.abf


In [46]:
# Exercise 3: what studies a given brain region?

import fairgraph.openminds.sands as sands

# the same region name occurs in several parcellations, so ask for all of them
regions = sands.ParcellationEntity.by_name("hippocampal formation", client, all=True)
print(f"{len(regions)} parcellation entities named 'hippocampal formation'")

region = regions[0]
print(f"using {region.name} ({region.uuid})")
print()

# count() takes the same filters as list(), without fetching the results
for cls in classes_of_interest:
    print(f"{cls.__name__:<16} {cls.count(client, study_targets=region):>4}")

3 parcellation entities named 'hippocampal formation'
using hippocampal formation (a656c10d-4a1c-4e57-950c-0526a4ad05ce)



DatasetVersion      1


Model               0


Simulation          0


DataAnalysis        0


ValidationTest      0
